In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline

In [ ]:
PROJECT_ROOT=Path("../../").resolve()
DATA_PATH = PROJECT_ROOT/"ml"/"outputs"/"model_features.csv"
MODEL_DIR = PROJECT_ROOT/"ml"/"models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "isolation_forest.pkl"

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
invoice_ids = df["invoice_id"].copy()
X = df.drop(columns=["invoice_id"])
print("X shape:", X.shape)
print("Number of features:", X.shape[1])

In [ ]:
X = X.apply(pd.to_numeric, errors="coerce")


In [ ]:
missing_values = X.isna().sum()
missing_values[missing_values > 0]

In [ ]:
infinite_values = np.isinf(X).sum()
infinite_values[infinite_values > 0]

In [ ]:
X = X.replace([np.inf, -np.inf], np.nan)
print("Infinite values handled.")

In [ ]:
preprocessor = SimpleImputer(
    strategy="median"
)

In [ ]:
isolation_forest = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    max_samples="auto",
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

In [ ]:
model_pipeline = Pipeline([
    ("imputer", preprocessor),
    ("isolation_forest", isolation_forest)
])

In [ ]:
model_pipeline.fit(X)

In [ ]:
predictions = model_pipeline.predict(X)


In [ ]:
raw_scores = model_pipeline.decision_function(X)
print(raw_scores[:10])

In [ ]:
anomaly_scores = -raw_scores

In [ ]:
results = pd.DataFrame({
    "invoice_id": invoice_ids,
    "anomaly_score": anomaly_scores,
    "is_anomaly": predictions == -1
})
results.head()

In [ ]:
anomaly_count = results["is_anomaly"].sum()
total_count = len(results)

anomaly_percentage = (
    anomaly_count / total_count
) * 100

print("Total invoices :", total_count)
print("Anomalies      :", anomaly_count)
print("Anomaly %      :", round(anomaly_percentage, 2))

In [ ]:
results["anomaly_score"].describe()

In [ ]:
high_threshold = results["anomaly_score"].quantile(0.95)
medium_threshold = results["anomaly_score"].quantile(0.80)

def assign_risk(score):
    if score >= high_threshold:
        return "HIGH"
    elif score >= medium_threshold:
        return "MEDIUM"
    else:
        return "LOW"

results["ml_risk"] = results["anomaly_score"].apply(assign_risk)

In [ ]:
results["rank_percentile"] = (
    results["anomaly_score"]
    .rank(pct=True) * 100
)

In [ ]:
top_anomalies = (
    results
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(20)
)

top_anomalies

In [ ]:
results["ml_risk"].value_counts()
results["ml_risk"].value_counts(normalize=True) * 100

In [ ]:
ANOMALY_OUTPUT = PROJECT_ROOT/"ml"/"outputs"/"anomaly_scores.csv"
results.to_csv(
    ANOMALY_OUTPUT,
    index=False
)


In [ ]:
ANOMALY_OUTPUT = PROJECT_ROOT / "ml" / "outputs" / "anomaly_scores.csv"

results.to_csv(
    ANOMALY_OUTPUT,
    index=False
)

print("Saved:", ANOMALY_OUTPUT)

In [ ]:
joblib.dump(
    model_pipeline,
    MODEL_PATH
)

print("Model saved:", MODEL_PATH)

In [ ]:
saved_model = joblib.load(MODEL_PATH)
print(type(saved_model))
print(saved_model.named_steps)

In [ ]:
test_predictions = saved_model.predict(X.head(10))
print(test_predictions)

In [ ]:
test_scores = -saved_model.decision_function(X.head(10))
print(test_scores)

In [ ]:
print("=" * 60)
print("ISOLATION FOREST TRAINING SUMMARY")
print("=" * 60)

print(f"Total invoices       : {len(results):,}")
print(f"Features used        : {X.shape[1]}")
print(f"Anomalies detected   : {anomaly_count:,}")
print(f"Anomaly percentage   : {anomaly_percentage:.2f}%")
print(f"High risk invoices   : {(results['ml_risk'] == 'HIGH').sum():,}")
print(f"Medium risk invoices : {(results['ml_risk'] == 'MEDIUM').sum():,}")
print(f"Low risk invoices    : {(results['ml_risk'] == 'LOW').sum():,}")

